# Multimodal BERTopic for Venue Topic Modeling

This notebook extracts semantic topics from venue reviews using BERTopic with:
- Sentence-BERT for text embeddings
- CLIP for image embeddings (optional)
- UMAP for dimensionality reduction
- K-Means/HDBSCAN for clustering
- c-TF-IDF for topic representation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import get_config, PROCESSED_DATA_DIR, MODEL_DIR
from src.models.bertopic import (
    TextEmbedder,
    ImageEmbedder,
    MultimodalEmbedder,
    VenueTopicExtractor,
    GeoClusterer,
    TimeContextEncoder,
    cluster_venues_by_location,
)
from src.utils.helpers import set_seed

# Setup
set_seed(42)
config = get_config()

# Output directory
OUTPUT_DIR = MODEL_DIR / 'bertopic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
# Load reviews
reviews_path = PROCESSED_DATA_DIR / 'train_reviews.parquet'
businesses_path = PROCESSED_DATA_DIR / 'businesses.parquet'

if reviews_path.exists():
    reviews_df = pd.read_parquet(reviews_path)
    print(f"Loaded {len(reviews_df)} reviews")
else:
    raise FileNotFoundError(f"Reviews not found. Run data preparation first.")

if businesses_path.exists():
    businesses_df = pd.read_parquet(businesses_path)
    print(f"Loaded {len(businesses_df)} businesses")
else:
    businesses_df = None
    print("No business data found")

In [ ]:
# Check columns
print("Review columns:", reviews_df.columns.tolist())
if businesses_df is not None:
    print("Business columns:", businesses_df.columns.tolist())

In [ ]:
# Determine text column
text_col = 'clean_text' if 'clean_text' in reviews_df.columns else 'text'
print(f"Using text column: {text_col}")

# Sample review
print(f"\nSample review:\n{reviews_df[text_col].iloc[0][:500]}...")

## 2. Text Embeddings with Sentence-BERT

In [ ]:
# Initialize text embedder
text_embedder = TextEmbedder(model_name='all-MiniLM-L6-v2')
print(f"Text embedding dimension: {text_embedder.embedding_dim}")

In [ ]:
# Test embedding generation
sample_texts = reviews_df[text_col].head(5).tolist()
sample_embeddings = text_embedder.encode(sample_texts)

print(f"Sample embeddings shape: {sample_embeddings.shape}")

In [ ]:
# Group reviews by venue
venue_reviews = reviews_df.groupby('business_id')[text_col].apply(list).to_dict()
venue_ids = list(venue_reviews.keys())

print(f"Number of venues: {len(venue_ids)}")
print(f"Average reviews per venue: {np.mean([len(r) for r in venue_reviews.values()]):.1f}")

In [ ]:
# Generate aggregated venue embeddings
# This may take a while for large datasets
venue_texts = [venue_reviews[vid] for vid in venue_ids]

# Limit for demo - use first 1000 venues
MAX_VENUES = 1000
if len(venue_ids) > MAX_VENUES:
    print(f"Limiting to {MAX_VENUES} venues for demo")
    venue_ids = venue_ids[:MAX_VENUES]
    venue_texts = venue_texts[:MAX_VENUES]

venue_embeddings = text_embedder.encode_aggregated(
    venue_texts,
    aggregation='mean',
    batch_size=32,
)

print(f"Venue embeddings shape: {venue_embeddings.shape}")

## 3. Topic Extraction with BERTopic

In [ ]:
# Configure topic extractor
config.bertopic.kmeans_n_clusters = 30  # Number of topics
config.bertopic.umap_n_components = 5

# Create extractor
extractor = VenueTopicExtractor(config=config.bertopic)
extractor.build_model(use_kmeans=True)  # Use K-Means for fixed number of topics

In [ ]:
# Prepare documents (concatenated reviews per venue)
documents = [" ".join(reviews[:10]) for reviews in venue_texts]  # Limit reviews per venue

print(f"Number of documents: {len(documents)}")
print(f"Average document length: {np.mean([len(d.split()) for d in documents]):.0f} words")

In [ ]:
# Fit topic model
topics, probs = extractor.fit(documents, embeddings=venue_embeddings)

print(f"\nTopics assigned: {len(topics)}")
print(f"Unique topics: {len(set(topics))}")

In [ ]:
# Get topic information
topic_info = extractor.get_topic_info()

print(f"\nExtracted {len(topic_info)} topics:\n")
for info in topic_info[:15]:
    print(f"Topic {info.topic_id}: {info.name}")
    print(f"  Count: {info.count} venues")
    print(f"  Keywords: {', '.join(info.representation[:5])}")
    print()

In [ ]:
# Topic distribution
topic_counts = pd.Series(topics).value_counts().sort_index()

plt.figure(figsize=(12, 5))
topic_counts[topic_counts.index >= 0].plot(kind='bar')
plt.xlabel('Topic ID')
plt.ylabel('Number of Venues')
plt.title('Venue Distribution Across Topics')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Topic Visualizations

In [ ]:
# Interactive topic visualization
try:
    fig = extractor.visualize_topics()
    fig.show()
except Exception as e:
    print(f"Visualization failed: {e}")

In [ ]:
# Topic barchart
try:
    fig = extractor.visualize_barchart(top_n_topics=10)
    fig.show()
except Exception as e:
    print(f"Barchart failed: {e}")

## 5. Geographic Clustering

In [ ]:
# Check if we have location data
if businesses_df is not None and 'latitude' in businesses_df.columns:
    print("Location data available")
    print(f"Latitude range: {businesses_df['latitude'].min():.2f} to {businesses_df['latitude'].max():.2f}")
    print(f"Longitude range: {businesses_df['longitude'].min():.2f} to {businesses_df['longitude'].max():.2f}")
else:
    print("No location data available")

In [ ]:
if businesses_df is not None and 'latitude' in businesses_df.columns:
    # Cluster venues by location
    venue_regions, geo_clusterer = cluster_venues_by_location(
        businesses_df,
        lat_col='latitude',
        lon_col='longitude',
        id_col='business_id',
        min_cluster_size=10,
    )
    
    print(f"\nFound {len(geo_clusterer.regions)} geographic regions:\n")
    for region in geo_clusterer.regions[:10]:
        print(f"{region.name}:")
        print(f"  Venues: {region.venue_count}")
        print(f"  Center: ({region.center_lat:.4f}, {region.center_lon:.4f})")
        print(f"  Radius: {region.radius_km:.2f} km")
        print()

In [ ]:
# Visualize geographic clusters
if businesses_df is not None and 'latitude' in businesses_df.columns:
    # Merge with cluster labels
    plot_df = businesses_df.merge(
        venue_regions[['venue_id', 'region_id', 'region_name']],
        left_on='business_id',
        right_on='venue_id',
        how='left'
    )
    
    plt.figure(figsize=(12, 8))
    
    # Plot outliers in gray
    outliers = plot_df[plot_df['region_id'] == -1]
    plt.scatter(
        outliers['longitude'], 
        outliers['latitude'],
        c='lightgray',
        s=5,
        alpha=0.3,
        label='Outliers'
    )
    
    # Plot clusters
    clustered = plot_df[plot_df['region_id'] >= 0]
    scatter = plt.scatter(
        clustered['longitude'],
        clustered['latitude'],
        c=clustered['region_id'],
        cmap='tab20',
        s=10,
        alpha=0.6
    )
    
    plt.colorbar(scatter, label='Region ID')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Geographic Clustering of Venues')
    plt.tight_layout()
    plt.show()

## 6. Context Encoding (Time & Weather)

In [ ]:
# Time context encoder
time_encoder = TimeContextEncoder(embedding_dim=16)

# Example: encode different time contexts
hours = np.array([9, 14, 19, 23])  # Morning, afternoon, evening, night
months = np.array([3, 7, 10, 1])   # Spring, summer, fall, winter
is_weekend = np.array([False, True, False, True])

time_embeddings = time_encoder.encode(hours, months, is_weekend)

print("Time context embeddings:")
for i, (h, m, w) in enumerate(zip(hours, months, is_weekend)):
    day_type = "weekend" if w else "weekday"
    print(f"  {h}:00, Month {m}, {day_type}: shape {time_embeddings[i].shape}")

## 7. Create Venue-Topic DataFrame

In [ ]:
# Create venue topics DataFrame
venue_topics_df = pd.DataFrame({
    'venue_id': venue_ids,
    'topic': topics,
    'topic_prob': [p.max() if isinstance(p, np.ndarray) else p for p in probs],
})

# Add topic names
topic_names = {t.topic_id: t.name for t in topic_info}
venue_topics_df['topic_name'] = venue_topics_df['topic'].map(topic_names)

print(venue_topics_df.head(10))

In [ ]:
# Merge with geographic regions if available
if 'venue_regions' in dir():
    venue_topics_df = venue_topics_df.merge(
        venue_regions[['venue_id', 'region_id', 'region_name']],
        on='venue_id',
        how='left'
    )
    print("Added geographic regions")
    print(venue_topics_df.head())

## 8. Save Results

In [ ]:
# Save topic model
extractor.save(OUTPUT_DIR / 'topic_model')
print(f"Saved topic model to {OUTPUT_DIR / 'topic_model'}")

# Save venue topics
venue_topics_df.to_parquet(OUTPUT_DIR / 'venue_topics.parquet', index=False)
print(f"Saved venue topics to {OUTPUT_DIR / 'venue_topics.parquet'}")

# Save venue embeddings
np.save(OUTPUT_DIR / 'venue_embeddings.npy', venue_embeddings)
print(f"Saved venue embeddings to {OUTPUT_DIR / 'venue_embeddings.npy'}")

# Save topic info
topic_info_df = pd.DataFrame([
    {
        'topic_id': t.topic_id,
        'name': t.name,
        'count': t.count,
        'keywords': ', '.join(t.representation[:10]),
    }
    for t in topic_info
])
topic_info_df.to_csv(OUTPUT_DIR / 'topic_info.csv', index=False)
print(f"Saved topic info to {OUTPUT_DIR / 'topic_info.csv'}")

In [ ]:
# Save visualizations
try:
    extractor.visualize_topics(OUTPUT_DIR / 'topic_visualization.html')
    extractor.visualize_barchart(top_n_topics=20, output_path=OUTPUT_DIR / 'topic_barchart.html')
    print("Saved visualizations")
except Exception as e:
    print(f"Failed to save visualizations: {e}")

## Summary

We've extracted:
1. **Venue Topics**: Semantic themes from review text
2. **Geographic Regions**: Location-based clusters
3. **Context Encoders**: Time and weather embeddings

These will be used in Phase 4 to build the heterogeneous graph:
- **User nodes**: MBTI embeddings (from Phase 2)
- **Venue nodes**: Topic embeddings (from this phase)
- **Context nodes**: Region, time, weather embeddings